# LA-08 — Original 1:1 strategy × Claude’s Deribit methodology

**Question:** Does the original A03b two-touch / **unchanged structural target and mirrored 1:1 stop** behave differently when we add Claude’s **signal-time CVD divergence** or **causally available 18:00 New York volume-profile structure** on the **Deribit-native** 2024–2026 signal universe?

## Research contract

- **Original trade:** EXT A03b forward 1:1 **FIRST print after confirmed second-touch close**, same target, mirrored stop, 240-minute maximum hold; **no new replay or revised barrier**. All source outcomes and fills remain exactly the ones originally simulated by A03b.
- **Claude methodology:** EXT venue-native signal generation and historical period/day-cluster bootstrap; V2 signal-time `cvd_ok`, `cvd_flow`, `v2_target_source` produced with Claude’s original CVD + DST-aware 18:00 ET session profile. The profile condition here is **availability of a causal LVN→HVN target**, *not adoption of V2’s different target*.
- **Controlled four A03b policies:** all candidates / CVD-only gate / profile-only gate / both gates. Eligibility is determined **at the original signal close**, never by the exit outcome.
- **Separate reference:** Claude V2 **CVD ON + profile target + VD18 stop + TRAIL_1M (no CUT15)**, from V2’s pre-existing trade replay. Its TP, stop and management **differ** from A03b; do not attribute paired differences to CVD alone.
- **Costs:** Claude EXT fixed Deribit assumptions (5 bp taker/side plus preserved A03 adverse print-fill offsets); funding, bid/ask queue and portfolio overlapping positions are not modeled.
- **Not new OOS:** all historical EXT/V2 data have been inspected. H1/H2 “holdout” labels are Claude’s previous period definition, **not untouched holdout after our newer analyses**. The notebook can describe and reject hypotheses; it cannot establish a tradable edge.

Outputs: `08_policy_candidate_panel.csv`, `08_policy_comparison.csv`, `08_paired_selection_effects.csv`, `08_decision.json`, and `08_comparison_net_bp.png`. All files are under `notebooks/Liquidity_Absorption/08/`. No raw trades are copied or downloaded.

In [ ]:
from pathlib import Path
import sys, subprocess
import pandas as pd
from IPython.display import display, Image
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO/'src'/'data_engineering').is_dir():
    REPO = REPO.parent
assert (REPO/'src'/'data_engineering').is_dir(), 'Run Jupyter inside the repo'
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from src.data_engineering.modeling.liquidity_absorption_bridge_08 import VERSION, Spec, run, INPUTS
assert VERSION == 'LA-BRIDGE-08-0.1.0', 'git pull, restart kernel'
for rel in INPUTS.values():
    file = REPO/'notebooks'/'Liquidity_Absorption'/rel
    assert file.exists(), f'Missing Claude EXT/V2 output: {file}'
print('LA08:', VERSION, 'source results:', [str(s) for s in INPUTS.values()])
r = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                    '-p', 'test_liquidity_absorption_bridge_08.py', '-q'],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1500:],r.stderr[-3500:])
assert r.returncode == 0, 'LA08 regression tests failed; do not run historical study'
OUT = REPO/'notebooks'/'Liquidity_Absorption'/'08'


## 1. Reconcile signal IDs and execution geometry

The engine refuses to proceed if the original 1:1 geometry changed, candidate IDs duplicate, or the EXT and V2 signals disagree on side, market or UTC signal-close timestamp. This is essential: the comparison must refer to **the same Deribit-native original candidates**, not a loosely matched group of price events.

The four original 1:1 policies do not re-run the tape: their profit/loss on an eligible trade must be **identical to A03b**. The V2 reference uses its own previously executed print-by-print outcomes.

In [ ]:
report = run(REPO, OUT, Spec())
display(report)
panel = pd.read_csv(OUT/'08_policy_candidate_panel.csv')
summary = pd.read_csv(OUT/'08_policy_comparison.csv')
paired = pd.read_csv(OUT/'08_paired_selection_effects.csv')
print('Candidate accounting by policy:')
display(panel.groupby('policy').agg(signals=('a02_candidate_row','size'),
                                   eligible=('policy_eligible','sum'),
                                   entries=('executed_entry','sum'),
                                   closed=('policy_closed','sum'),
                                   unresolved=('policy_censored','sum')).reset_index())
assert panel.groupby('policy').a02_candidate_row.nunique().eq(report['original_signals']).all()


## 2. Claude H1 / H2 partitions — historical retrospective comparisons

Read **mean net bp per completed trade** separately from **realized closed-P&L contribution per original signal**. A filter can improve the former simply by trading less. The latter assigns zero only to trades never selected or not realized; **censored positions are not treated as known zero-P&L exits**. Consequently this is **not full portfolio expectancy**.

Confidence bounds cluster by UTC signal day (2,000 resamples). H1 and H2 were part of Claude’s earlier preregistration; now both are previously inspected, so no fresh-confirmation language is appropriate.

In [ ]:
columns=['policy','symbol','period','n_original_signals','n_eligible','n_closed',
         'n_censored','n_target','n_stop','n_timeout',
         'mean_net_bp_per_closed_trade','net_closed_day_boot_lo95',
         'realized_closed_net_bp_per_original_signal']
display(summary.loc[summary.period.eq('HOLDOUT_INSPECTED'),columns].round(3))
print('Paired realized-contribution change vs unchanged A03b, same candidates:')
display(paired[paired.period.eq('HOLDOUT_INSPECTED')].round(3))


## 3. Inspect losses, not just conditional hit rate

Note how many signals are skipped by the V2 profile rule and whether the filtered candidate set differs in the base strategy’s stop, timeout or target shares. Examine BTC/ETH and H1/H2 separately. A result exceeding zero only within an inspected historical subset does **not** override prior negative EXT/V2 tests.

**Critical attribution:** original A03b uses the existing VWAP/prior-day structural TP and its mirrored stop; Claude V2 instead uses a next-HVN target, VD18 volatility stop, and trailing. Their economic outcomes cannot be attributed solely to any one condition without a further frozen one-change-at-a-time tape replay.

In [ ]:
plot = OUT/'08_comparison_net_bp.png'
display(Image(filename=str(plot)))
print('Outcome counts by policy, including all skipped/censored original IDs:')
display(panel.groupby(['policy','policy_outcome']).size().rename('signals').reset_index())


## 4. Research decision

This notebook is an **auditably controlled retrospective bridge**, not a new live-trading model. The old A03b 1:1 contract remains untouched; CVD and profile flags are copied from Claude V2 signals, which are anchored to the original signal and do not use future returns. The Claude V2 full implementation is included strictly as a separate reference, not a matched one-factor experiment.

Before any prospective strategy trial, verify archive completeness and live fee schedules, agree tick-valid stops, reserve genuinely later unseen dates, account for concurrent trade exposure, and preregister at most one simple gate. A negative conditional net result should not be treated as motivation to search in-sample thresholds.